# Persona Validation Results — Visual Explanation

This notebook explains the validation results from the persona-based answer generation pipeline.

It is organized into four layers:

1. **Overall quality** — how well the system performs across all answers
2. **Question-level performance** — which questions are easier or harder to predict
3. **Persona-level variation** — which personas are modeled better or worse
4. **Group and trend alignment** — whether the model preserves patterns by age, gender, and education

The notebook expects these files in the same directory as the notebook, or you can override the paths in the setup cell:

- `validation_summary.json`
- `persona_metrics.csv`
- `question_metrics.csv`
- `group_metrics.csv`
- `group_question_metrics.csv`
- `trend_metrics.csv`
- `trend_detail.csv`


In [ ]:

from pathlib import Path
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

CANDIDATE_DIRS = [
    Path("."),
    Path.cwd(),
    Path("/data"),
]

REQUIRED_FILES = [
    "validation_summary.json",
    "persona_metrics.csv",
    "question_metrics.csv",
    "group_metrics.csv",
    "group_question_metrics.csv",
    "trend_metrics.csv",
    "trend_detail.csv",
]

def pick_base_dir(candidates, required_files):
    for candidate in candidates:
        candidate = candidate.resolve()
        if all((candidate / f).exists() for f in required_files):
            return candidate
    for candidate in candidates:
        candidate = candidate.resolve()
        print(f"Checked {candidate} -> {[ (candidate / f).exists() for f in required_files ]}")
    raise FileNotFoundError(
        "Could not find all required files. Put the notebook next to the validation outputs "
        "or edit CANDIDATE_DIRS / PATHS below."
    )

BASE_DIR = pick_base_dir(CANDIDATE_DIRS, REQUIRED_FILES)

PATHS = {
    "summary": BASE_DIR / "validation_summary.json",
    "persona": BASE_DIR / "persona_metrics.csv",
    "question": BASE_DIR / "question_metrics.csv",
    "group": BASE_DIR / "group_metrics.csv",
    "group_question": BASE_DIR / "group_question_metrics.csv",
    "trend": BASE_DIR / "trend_metrics.csv",
    "trend_detail": BASE_DIR / "trend_detail.csv",
}

print(f"Using BASE_DIR = {BASE_DIR}")
for name, path in PATHS.items():
    print(f"{name:>14}: {path} | exists={path.exists()}")

with open(PATHS["summary"], "r", encoding="utf-8") as f:
    summary = json.load(f)

persona_df = pd.read_csv(PATHS["persona"])
question_df = pd.read_csv(PATHS["question"])
group_df = pd.read_csv(PATHS["group"])
group_question_df = pd.read_csv(PATHS["group_question"])
trend_df = pd.read_csv(PATHS["trend"])
trend_detail_df = pd.read_csv(PATHS["trend_detail"])

overall = pd.Series(summary["overall"]).to_frame("value")
overall


## 1) Overall validation summary

These metrics describe the model across all compared answers.

How to read the main metrics:

- **Exact match / F1**: higher is better
- **MAE / RMSE**: lower is better
- **Bias**: closer to 0 is better
- **Pearson / Spearman**: higher is better
- **TVD / JS divergence**: lower is better

A useful reading pattern is:
- first check **exact match** and **weighted F1** for answer correctness,
- then check **RMSE** and **bias** for score realism,
- then check **Pearson/Spearman** for ranking consistency,
- then check **TVD/JS** for distribution similarity.


In [ ]:

key_rows = [
    ("Compared answers", summary["overall"]["compared"]),
    ("Exact match", summary["overall"]["exact_match"]),
    ("Weighted F1", summary["overall"]["weighted_f1"]),
    ("MAE", summary["overall"]["mae"]),
    ("RMSE", summary["overall"]["rmse"]),
    ("Mean bias", summary["overall"]["mean_bias"]),
    ("Pearson", summary["overall"]["pearson"]),
    ("Spearman", summary["overall"]["spearman"]),
    ("TVD", summary["overall"]["tvd"]),
    ("JS divergence", summary["overall"]["js_divergence"]),
]
overall_table = pd.DataFrame(key_rows, columns=["Metric", "Value"])
display(overall_table.style.format({"Value": "{:.4f}"}))

fig, ax = plt.subplots(figsize=(11, 4))
plot_df = overall_table.copy()
plot_df = plot_df[plot_df["Metric"] != "Compared answers"]
ax.bar(plot_df["Metric"], plot_df["Value"])
ax.set_title("Overall validation metrics")
ax.set_ylabel("Value")
ax.tick_params(axis="x", rotation=45)
plt.show()


## 2) Question-level performance

This section highlights where the model is strong and where it struggles.

Recommended interpretation:
- Use **exact match** and **weighted F1** to find the most and least predictable questions
- Use **RMSE** and **mean bias** to see whether the model systematically overshoots or undershoots
- Use **Pearson/Spearman** to see where the ordering of responses is preserved even when exact answers are not


In [ ]:

display(question_df.sort_values("weighted_f1", ascending=False).reset_index(drop=True).style.format({
    "exact_match": "{:.3f}",
    "weighted_f1": "{:.3f}",
    "macro_f1": "{:.3f}",
    "mae": "{:.3f}",
    "rmse": "{:.3f}",
    "mean_bias": "{:.3f}",
    "pearson": "{:.3f}",
    "spearman": "{:.3f}",
    "tvd": "{:.3f}",
    "js_divergence": "{:.3f}",
}))

best_q = question_df.nlargest(5, "weighted_f1")[["question", "weighted_f1", "exact_match", "rmse"]]
worst_q = question_df.nsmallest(5, "weighted_f1")[["question", "weighted_f1", "exact_match", "rmse"]]

print("Best questions by weighted F1")
display(best_q.style.format({"weighted_f1": "{:.3f}", "exact_match": "{:.3f}", "rmse": "{:.3f}"}))

print("Worst questions by weighted F1")
display(worst_q.style.format({"weighted_f1": "{:.3f}", "exact_match": "{:.3f}", "rmse": "{:.3f}"}))


In [ ]:

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

tmp = question_df.sort_values("weighted_f1", ascending=True)
axes[0, 0].barh(tmp["question"], tmp["weighted_f1"])
axes[0, 0].set_title("Weighted F1 by question")
axes[0, 0].set_xlabel("Weighted F1")

tmp = question_df.sort_values("exact_match", ascending=True)
axes[0, 1].barh(tmp["question"], tmp["exact_match"])
axes[0, 1].set_title("Exact match by question")
axes[0, 1].set_xlabel("Exact match")

tmp = question_df.sort_values("rmse", ascending=False)
axes[1, 0].barh(tmp["question"], tmp["rmse"])
axes[1, 0].set_title("RMSE by question")
axes[1, 0].set_xlabel("RMSE")

tmp = question_df.sort_values("mean_bias")
axes[1, 1].barh(tmp["question"], tmp["mean_bias"])
axes[1, 1].axvline(0, linestyle="--", linewidth=1)
axes[1, 1].set_title("Mean bias by question")
axes[1, 1].set_xlabel("Predicted mean - actual mean")

plt.tight_layout()
plt.show()


## 3) Persona-wise validation

This view answers a practical question:

**Are errors spread evenly across personas, or is performance highly uneven?**

What to look for:
- A wide RMSE spread means some personas are much harder to simulate than others
- A wide bias spread means some personas are systematically over- or under-predicted
- Grouped boxplots show whether errors are concentrated in specific demographic slices


In [ ]:

metric_cols = ["exact_match", "weighted_f1", "mae", "rmse", "mean_bias", "pearson", "spearman"]
display(persona_df[["PersonaId", "Q29", "Q30", "Q31"] + metric_cols].head())

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].hist(persona_df["exact_match"].dropna(), bins=20)
axes[0, 0].set_title("Distribution of persona exact match")
axes[0, 0].set_xlabel("Exact match")

axes[0, 1].hist(persona_df["rmse"].dropna(), bins=20)
axes[0, 1].set_title("Distribution of persona RMSE")
axes[0, 1].set_xlabel("RMSE")

axes[1, 0].scatter(persona_df["exact_match"], persona_df["rmse"], alpha=0.6)
axes[1, 0].set_title("Persona exact match vs RMSE")
axes[1, 0].set_xlabel("Exact match")
axes[1, 0].set_ylabel("RMSE")

axes[1, 1].hist(persona_df["mean_bias"].dropna(), bins=20)
axes[1, 1].axvline(0, linestyle="--", linewidth=1)
axes[1, 1].set_title("Distribution of persona mean bias")
axes[1, 1].set_xlabel("Bias")

plt.tight_layout()
plt.show()

print("Best personas by exact match")
display(persona_df.nlargest(10, "exact_match")[["PersonaId", "Q29", "Q30", "Q31", "exact_match", "weighted_f1", "rmse"]]
        .style.format({"exact_match":"{:.3f}", "weighted_f1":"{:.3f}", "rmse":"{:.3f}"}))

print("Worst personas by exact match")
display(persona_df.nsmallest(10, "exact_match")[["PersonaId", "Q29", "Q30", "Q31", "exact_match", "weighted_f1", "rmse"]]
        .style.format({"exact_match":"{:.3f}", "weighted_f1":"{:.3f}", "rmse":"{:.3f}"}))


In [ ]:

def simple_boxplot(df, group_col, value_col, title, max_groups=12):
    plot_df = df[[group_col, value_col]].dropna().copy()
    counts = plot_df[group_col].value_counts()
    keep = counts.head(max_groups).index
    plot_df = plot_df[plot_df[group_col].isin(keep)]
    groups = [plot_df.loc[plot_df[group_col] == g, value_col].values for g in keep]
    plt.figure(figsize=(12, 5))
    plt.boxplot(groups, labels=list(keep), vert=True)
    plt.title(title)
    plt.ylabel(value_col)
    plt.xticks(rotation=30)
    plt.show()

simple_boxplot(persona_df, "Q29", "rmse", "Persona RMSE by age group")
simple_boxplot(persona_df, "Q30", "rmse", "Persona RMSE by gender")
simple_boxplot(persona_df, "Q31", "rmse", "Persona RMSE by education")


## 4) Group-wise validation

This section compresses the results by demographic group:
- **Q29** = age
- **Q30** = gender
- **Q31** = education

This answers whether the model performs better for some population slices than others.


In [ ]:

group_label_map = {"Q29": "Age", "Q30": "Gender", "Q31": "Education"}
group_df["group_label"] = group_df["group_column"].map(group_label_map).fillna(group_df["group_column"])

display(group_df.sort_values(["group_column", "group_value"]).style.format({
    "exact_match": "{:.3f}",
    "weighted_f1": "{:.3f}",
    "mae": "{:.3f}",
    "rmse": "{:.3f}",
    "mean_bias": "{:.3f}",
    "pearson": "{:.3f}",
    "spearman": "{:.3f}",
    "tvd": "{:.3f}",
    "js_divergence": "{:.3f}",
}))

for metric in ["exact_match", "rmse", "pearson", "mean_bias"]:
    plt.figure(figsize=(12, 5))
    plot_df = group_df.sort_values(["group_column", "group_value"]).copy()
    labels = plot_df["group_label"] + ": " + plot_df["group_value"].astype(str)
    plt.bar(labels, plot_df[metric])
    if metric == "mean_bias":
        plt.axhline(0, linestyle="--", linewidth=1)
    plt.title(f"{metric} by demographic group")
    plt.xticks(rotation=60, ha="right")
    plt.tight_layout()
    plt.show()


## 5) Group × question heatmaps

These heatmaps show where performance breaks down in a more detailed way.

Suggested reading:
- **Exact match heatmap**: where the model gets answers right most often
- **Bias heatmap**: where the model systematically predicts too high or too low
- **RMSE heatmap**: where the model misses badly even if averages look acceptable


In [ ]:

def plot_heatmap(df, value_col, group_column, title, cmap="viridis"):
    sub = df[df["group_column"] == group_column].copy()
    pivot = sub.pivot(index="group_value", columns="question", values=value_col)
    fig, ax = plt.subplots(figsize=(14, max(4, len(pivot) * 0.5)))
    im = ax.imshow(pivot.values, aspect="auto", cmap=cmap)
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_title(title)
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label(value_col)
    plt.tight_layout()
    plt.show()

for group_column, label in [("Q29", "Age"), ("Q30", "Gender"), ("Q31", "Education")]:
    plot_heatmap(group_question_df, "exact_match", group_column, f"Exact match heatmap — {label}")
    plot_heatmap(group_question_df, "rmse", group_column, f"RMSE heatmap — {label}")
    plot_heatmap(group_question_df, "mean_bias", group_column, f"Mean bias heatmap — {label}", cmap="coolwarm")


## 6) Trend preservation across groups

A model can be weak on exact answers but still preserve **relative trends** across demographic groups.

This section checks:
- whether the predicted group means track the actual group means
- which questions preserve group structure well
- which questions distort the demographic pattern


In [ ]:

display(trend_df.sort_values(["group_column", "mean_absolute_gap_across_groups"]).style.format({
    "mean_absolute_gap_across_groups": "{:.3f}",
    "rmse_across_group_means": "{:.3f}",
    "pearson_group_mean_correlation": "{:.3f}",
    "spearman_group_mean_correlation": "{:.3f}",
}))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

tmp = trend_df.sort_values("mean_absolute_gap_across_groups", ascending=False)
axes[0].barh(
    tmp["group_column"] + " / " + tmp["question"],
    tmp["mean_absolute_gap_across_groups"]
)
axes[0].set_title("Mean absolute gap across group means")
axes[0].set_xlabel("Gap")

tmp = trend_df.sort_values("pearson_group_mean_correlation")
axes[1].barh(
    tmp["group_column"] + " / " + tmp["question"],
    tmp["pearson_group_mean_correlation"]
)
axes[1].axvline(0, linestyle="--", linewidth=1)
axes[1].set_title("Correlation of predicted vs actual group means")
axes[1].set_xlabel("Pearson correlation")

plt.tight_layout()
plt.show()


In [ ]:

def plot_group_trend(question, group_column):
    sub = trend_detail_df[(trend_detail_df["question"] == question) & (trend_detail_df["group_column"] == group_column)].copy()
    if sub.empty:
        print(f"No data for question={question}, group_column={group_column}")
        return
    x = np.arange(len(sub))
    plt.figure(figsize=(10, 4))
    plt.plot(x, sub["actual_mean"], marker="o", label="Actual mean")
    plt.plot(x, sub["predicted_mean"], marker="o", label="Predicted mean")
    plt.xticks(x, sub["group_value"], rotation=30, ha="right")
    plt.title(f"Group trend comparison — {group_column} / {question}")
    plt.ylabel("Mean response")
    plt.legend()
    plt.tight_layout()
    plt.show()

# Auto-select interesting examples:
worst_trends = trend_df.sort_values("mean_absolute_gap_across_groups", ascending=False).head(3)
best_trends = trend_df.sort_values("mean_absolute_gap_across_groups", ascending=True).head(3)

print("Worst preserved trends")
display(worst_trends)

for _, row in worst_trends.iterrows():
    plot_group_trend(row["question"], row["group_column"])

print("Best preserved trends")
display(best_trends)

for _, row in best_trends.iterrows():
    plot_group_trend(row["question"], row["group_column"])


## 7) Executive summary helper

This final cell creates a compact summary you can paste into slides or a report.


In [ ]:

overall = summary["overall"]
best_question = question_df.sort_values("weighted_f1", ascending=False).iloc[0]
worst_question = question_df.sort_values("weighted_f1", ascending=True).iloc[0]
best_group = group_df.sort_values("exact_match", ascending=False).iloc[0]
worst_group = group_df.sort_values("exact_match", ascending=True).iloc[0]
worst_trend = trend_df.sort_values("mean_absolute_gap_across_groups", ascending=False).iloc[0]

report_lines = [
    "# Compact narrative summary",
    "",
    f"- The validation compares **{summary['num_personas']} personas** across **{summary['num_questions']} target questions**.",
    f"- Overall exact match is **{overall['exact_match']:.3f}** and weighted F1 is **{overall['weighted_f1']:.3f}**.",
    f"- Numeric error is **MAE {overall['mae']:.3f}** and **RMSE {overall['rmse']:.3f}**.",
    f"- Average bias is **{overall['mean_bias']:.3f}**, meaning predictions are slightly {'lower' if overall['mean_bias'] < 0 else 'higher'} than the survey answers on average.",
    f"- The strongest question by weighted F1 is **{best_question['question']}** ({best_question['weighted_f1']:.3f}).",
    f"- The weakest question by weighted F1 is **{worst_question['question']}** ({worst_question['weighted_f1']:.3f}).",
    f"- The best-performing demographic slice by exact match is **{best_group['group_column']} = {best_group['group_value']}** ({best_group['exact_match']:.3f}).",
    f"- The weakest-performing demographic slice by exact match is **{worst_group['group_column']} = {worst_group['group_value']}** ({worst_group['exact_match']:.3f}).",
    f"- The most distorted demographic trend is **{worst_trend['group_column']} / {worst_trend['question']}**, with mean absolute gap **{worst_trend['mean_absolute_gap_across_groups']:.3f}**.",
]
display(Markdown("\n".join(report_lines)))
